In [2]:
%reset -s -f
%load_ext autoreload
%autoreload 2

from sympy.combinatorics import Permutation
from sympy import factorial
import random

from IPython.display import display, Math

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Maximum uncertainty Secret Santa algorithm

The properties of the algorithm (in the case of more than three participants) is that
* After the drawing, every other participant will be equally probable to be the giver, including the acknowledged gift recipient as well.
* The expected surprise is maximal in that case, if the identities are revealed at the end, just as in that game variant.
  
  Actually, with five participants, the expected surprise is 1% higher than with the original algorithm &#x1F600;.

About the Secret Santa tradition: https://en.wikipedia.org/wiki/Secret_Santa

#

$D_n$ : number of derangements of an n-element set

https://en.wikipedia.org/wiki/Derangement

In [3]:
def NumDerangements(n: int):
    memo = [-1] * (n + 1)

    return NumDerangementsMemo(memo, n)


def NumDerangementsMemo(memo: list, n: int):
    if n < 2:
        return 1 - n
    
    elif memo[n] != -1:
        return memo[n]
    
    memo[n] = (n-1) * (NumDerangementsMemo(memo, n-1) + NumDerangementsMemo(memo, n-2))
    return memo[n]

$Q_{n,k}$ : number of derangements of an n-element set with k 2-cycles

Recursive formula of $Q_{n,k}$:

$
    Q_{n, k} = \binom{n}{2}\binom{n-2}{2}\dots\binom{n-2(k-1)}{2} {1 \over k!} (D_{n - 2k} - \sum_{j > 0} Q_{n-2k, j}) \\
    \qquad = {n! \over 2^{k} (n-2(k-1))! k!} (D_{n - 2k} - \sum_{j > 0} Q_{n-2k, j})
$

In [4]:
def NumDerangementWith2Cycles(n: int, num2cycles: int):
    memo = [-1] * (n + 1)
    
    return NumDerangementsWith2CyclesMemo(memo, n, num2cycles)

    
def NumDerangementsWith2CyclesMemo(memo, n: int, k: int):
    n2k = n - 2 * k
    if n2k < 0:
        return 0
    
    S = NumDerangements(n2k)
    max2cycles = n2k // 2 - n2k % 2

    for j in range(1, max2cycles + 1):
        S = S - NumDerangementsWith2CyclesMemo(memo, n2k, j)

    m = max(2, n - (k - 1) * 2)
    F = factorial(n) / (2**k * factorial(k) * factorial(m - 2))
    memo[n] = F * S

    return memo[n]

In [5]:
# Detect if derangement has 2-cycles
def DerangementHasTwoCycles(p: Permutation):
    a = (p * p).array_form
    for i in range(len(a)):
        if a[i] == i:
            return True
    return False

In [6]:
def RandomDerangement(n: int, require2cycle=False):
    if n <= 1:
        return []
    
    while True:
        h = list(range(n))
        RandomDerangementRec(n, h)
        if not require2cycle or DerangementHasTwoCycles(Permutation(h)):
            break

    return h

def RandomDerangementRec(n: int, h: list):
    if n <= 1:
        return
    
    # P(n-1) receives h(i)
    i = random.randrange(n-1)
    p = random.uniform(0, 1)
    # P(i) receives h(n-1)
    if p < (n - 1) * NumDerangements(n - 2) / NumDerangements(n):
        h[i], h[n-1] = h[n-1], h[i]
        h[i], h[n-2] = h[n-2], h[i]
        RandomDerangementRec(n - 2, h)
        h[i], h[n-2] = h[n-2], h[i]
    # P(i) not receives h(n-1)
    else:
        hi = h[i]
        RandomDerangementRec(n - 1, h)
        idx = h.index(hi)
        h[n-1], h[idx] = hi, h[n-1]

Notations:

$
A \rightarrow B \text{ : person A give gift to person B} \\
\eta \text{ : number of 2-cycles in the derangement } \\ 
$

The probabilities of the required events:

$
P(B \rightarrow A, A \rightarrow B) = { D_{n-2} \over D_{n}} \\
P(A \rightarrow B) = { 1 \over n-1} \\
P(B \rightarrow A | A \rightarrow B) = (n-1) { D_{n-2} \over D_{n}} \\
P(C \rightarrow A | A \rightarrow B) = { 1 \over n - 2}(1 - P(B \rightarrow A | A \rightarrow B)) = { n-1 \over n - 2} { D_{n-1} \over D_{n}} \\
P(\eta = 0) = { Q_{n,0} \over D_{n} } \\
P(B \rightarrow A | A \rightarrow B, \eta \gt 0) = { P(B \rightarrow A | A \rightarrow B) \over 1 - P(\eta = 0)}\\
$

Make the conditional distribution uniform by "mixing" two (overlapping) events:

$
p_1 := P(B \rightarrow A | A \rightarrow B, \eta \gt 0) \\
p_2 :=P(B \rightarrow A | A \rightarrow B) \\
p := P(A \rightarrow B) \\
$

$
r p_1 + (1 - r) p_2 = p \\
r = { p - p_2 \over p_1 - p_2}\\
$

In [7]:
def BalancedRandomDerangement(n: int):
    # There is no need for correction (n = 4), or nothing can be done
    if n < 5:
        return RandomDerangement(n)
    
    p_AB_cond_A2B = (n - 1) * NumDerangements(n - 2) / NumDerangements(n)
    p_no2cycles = NumDerangementWith2Cycles(n, 0) / NumDerangements(n)
    
    p_AB_cond_A2B_2cycle = p_AB_cond_A2B / (1 - p_no2cycles)

    dp = p_AB_cond_A2B_2cycle - p_AB_cond_A2B
    assert(dp > 0)
    r = (1/(n-1) - p_AB_cond_A2B) / (p_AB_cond_A2B_2cycle - p_AB_cond_A2B) if dp > 1e-5 else 0

    p = random.uniform(0, 1)
    if p < r:   
        return RandomDerangement(n, require2cycle=True)
    else:
        return RandomDerangement(n)

In [8]:
# Simple simulation for P(A <-> B | A -> B) by rejection
def test(n: int, num_samples: int, A: int, B: int):
    for i in range(num_samples):
        while True:
            #d = Permutation(RandomDerangement(n))
            d = Permutation(BalancedRandomDerangement(n))
            B1 = A^d
            if B1 == B:
                break
        yield A^~d

In [9]:
num_samples = 10000
n = 6
A, B = 0, 1

# A <-> B | A -> B
s = [1 if S == B else 0 for S in test(n, num_samples, A, B)]
sum_s = sum(s)
p = sum_s/num_samples

n, p, p / (1/(n - 1))

(6, 0.2054, 1.027)

Using the formula (for $n \gt 0$):

$
D_n = n D_{n-1} + (-1)^n
$

The ratio of the conditionals:

$
u_n = { P(C \rightarrow A | A \rightarrow B) \over P(B \rightarrow A | A \rightarrow B) }
= { { n-1 \over n - 2} { D_{n-1} \over D_{n}} \over (n-1) { D_{n-2} \over D_{n}} } = { D_{n-1} \over (n-2) D_{n-2} }
$

$
u_{n+1} = { D_{n} \over (n-1) D_{n-1} } = { n D_{n-1} + (-1)^n \over (n-1) D_{n-1} } = {n \over n-1} + {(-1)^n \over (n-1)D_{n-1}} \xrightarrow[n\to\infty]{}1
$

Therefore, for a sufficiently large $n$, we obtain an approximate uniform distribution even with the original algorithm.